# Task A (CXR / ResNet18) — Colab 実行インターフェース

このNotebookは**実行のための薄い層**です。学習ロジックは `src/covid_mortality/` と `scripts/` にあり、
Colab以外のCUDA環境でも同じコマンドで再実行できます。

| 手順 | セル |
|---|---|
| Drive mount | 1 |
| GPU確認 | 2 |
| package install | 3 |
| コード取得（Drive上のZIPを `/content/code` へ展開。ZIP自体は変更しない） | 4 |
| config指定 | 5 |
| smoke test | 6 |
| 本学習（完了runは自動skip、切断後は再実行で再開） | 7 |
| 結果確認 | 8 |

**配置**
- コード：`00_共通・研究管理/AIagent_code/AIagent_code_20260920.zip`（読み取りのみ）
- 前処理キャッシュ：`01_TaskA_CXR/01_前処理/AIagent_taskA_png512_16bit`（読み取りのみ）
- run出力：`01_TaskA_CXR/04_Training/AIagent_taskA_runs`（ここにだけ書き込む）

**固定条件**：cohort 1,277 / Train 1,021・Val 128・Test 128 / 死亡 135・17・17。splitは再作成しない。
**Testはこのnotebookでは一切読み込みません**（最終評価は凍結後の別スクリプト）。

In [ ]:
# 1. Google Drive mount
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/気合のCOVID19'
TASKA = f'{DRIVE_ROOT}/01_TaskA_CXR/01_前処理'
CACHE = f'{TASKA}/AIagent_taskA_png512_16bit'      # 前処理キャッシュ（既存 processed_png_1277 とは別物）
# run出力（checkpoint / history / 予測）。既存Task Aの成果物は参照も上書きもしない。
RUNS_ROOT = f'{DRIVE_ROOT}/01_TaskA_CXR/04_Training/AIagent_taskA_runs'
import os
os.makedirs(RUNS_ROOT, exist_ok=True)
print('cache :', os.listdir(CACHE))
print('runs  :', RUNS_ROOT)

In [ ]:
# 2. GPU確認（T4以上を想定。GPUが無い場合はランタイムのタイプを変更する）
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# 3. package install（Colabには torch / torchvision / pandas / numpy / PIL が既定で入っている）
!pip -q install pydicom==3.0.2
import numpy, pandas, PIL, torchvision
print('numpy', numpy.__version__, '| pandas', pandas.__version__,
      '| torchvision', torchvision.__version__, '| Pillow', PIL.__version__)

In [ ]:
# 4. コード取得：Drive 上の ZIP を /content/code に展開して実行する
#    Drive 上の ZIP は読み取るだけで、変更も再作成もしない。
#    ZIP は scripts/11_build_code_bundle.py が POSIX ("/") 区切りで作成し、
#    同じ場所に <zip名>_manifest.json（期待 SHA256 を含む）を出力する。
import hashlib, json, shutil, zipfile
from pathlib import Path

ZIP_NAME = 'AIagent_code_20260921i.zip'      # 使う版に合わせて変更する
CODE_ZIP = Path(f'{DRIVE_ROOT}/00_共通・研究管理/AIagent_code/{ZIP_NAME}')
CODE = '/content/code'
EXPECTED_SHA256_FALLBACK = None              # manifest が無い場合のみ使う固定値

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

assert CODE_ZIP.exists(), f'ZIP が見つかりません: {CODE_ZIP}'
manifest = CODE_ZIP.with_name(CODE_ZIP.stem + '_manifest.json')
expected = (json.loads(manifest.read_text(encoding='utf-8'))['sha256'] if manifest.exists()
            else EXPECTED_SHA256_FALLBACK)
digest = sha256(CODE_ZIP)
print('zip sha256:', digest)
print('expected  :', expected, '(manifest)' if manifest.exists() else '(固定値)')
assert expected is not None, 'manifest が無く、固定値も設定されていません'
assert digest == expected, 'ZIP が配布時と異なります。配置したファイルを確認してください。'

# 展開前に、ZIP内部のentry名がPOSIX区切りであることを確認する（Linuxでの展開が壊れないため）
with zipfile.ZipFile(CODE_ZIP) as z:
    raw = [i.orig_filename for i in z.infolist()]
    bad = [n for n in raw if '\\' in n]
    assert not bad, f'ZIP内のentry名にバックスラッシュがあります: {bad[:3]}'
    print('entries:', len(raw), '| backslash entries:', len(bad))

# 毎回まっさらに展開する（前回の残骸が混ざらないようにする）。書き込みは /content のみ。
shutil.rmtree(CODE, ignore_errors=True)
Path(CODE).mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(CODE_ZIP) as z:
    z.extractall(CODE)

%cd {CODE}
for p in ['src/covid_mortality/training/taskA_trainer.py',
          'src/covid_mortality/evaluation/metrics.py',
          'src/covid_mortality/data/taskA_dataset.py',
          'src/covid_mortality/models/taskA_resnet18.py',
          'scripts/09_taskA_run_training.py',
          'scripts/12_taskA_stage1_report.py',
          'data/splits/COVID19_固定患者split_1277.csv',
          'data/interim/cxr_audit/index_cxr_manifest_window_T0m2_T0.csv']:
    assert Path(CODE, p).exists(), f'展開後に見つかりません: {p}'
assert not [p for p in Path(CODE).rglob('*') if '\\' in p.name], '展開結果にバックスラッシュ名のファイルがあります'
assert sha256(CODE_ZIP) == digest, 'Drive上のZIPが変化しました'
print('extracted to', CODE, '->', len(list(Path(CODE).rglob('*'))), 'entries; 必須ファイルを確認、ZIPは未変更')

In [ ]:
# 5. config指定（ここだけを編集する。学習ロジックは触らない）
IMAGE_DIR = f'{CACHE}/images'          # 1,277枚の16bit PNG
STAGE     = 'main'                     # main = 学習率4 × augmentation2 × seed3 = 24 run
MAX_EPOCHS   = 30
BATCH_SIZE   = 32
NUM_WORKERS  = 2
LIMIT        = 8                       # 1セッションで回すrun数。8 run単位で実行する。
                                       # 完了済みrunは自動skipなので、同じ値のまま3回実行すれば24 runになる。

# 画像はDriveから直接読むと遅いので、ローカルSSDへコピーして使う（Drive側は読むだけ）
LOCAL_IMAGES = '/content/images'
!mkdir -p {LOCAL_IMAGES} && rsync -a "{IMAGE_DIR}/" {LOCAL_IMAGES}/
print(len(os.listdir(LOCAL_IMAGES)), 'images ready')

In [ ]:
# 6. smoke test（2 run × 2 epoch、少数サンプル。本学習の前に配線を確認する）
!python scripts/07_taskA_dataset_qc.py --project . 2>&1 | tail -8
!python scripts/08_taskA_model_smoke.py --project . 2>&1 | tail -4
!python scripts/09_taskA_run_training.py --project . --smoke \
    --image-dir {LOCAL_IMAGES} --runs-root /content/smoke_runs --num-workers 0 2>&1 | tail -6

In [ ]:
# 7. 学習の実行
#    STAGE='main'   -> Stage 1（学習率4 × augmentation2 × seed3 = 24 run）
#    STAGE='stage2' -> Stage 2（warmup1 / staged × seed3 = 6 run。Stage 1 の run には触れない）
#    - 完了済みrunは自動でskip
#    - 各run終了ごとに checkpoint / history / val_predictions / DONE.json をDriveへ保存
#    - Colabが切断したら、このセルを再実行するだけで未完了runのみ再開する
#    - Testは読み込まない
STAGE = 'stage2'          # Stage 1 完了後は 'stage2' に変更する
LIMIT = None              # Stage 2 は 6 run。分割したい場合は 3 などにする

limit = f'--limit {LIMIT}' if LIMIT else ''
!python scripts/09_taskA_run_training.py --project . --stage {STAGE} \
    --image-dir {LOCAL_IMAGES} --runs-root "{RUNS_ROOT}" \
    --max-epochs {MAX_EPOCHS} --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} {limit}

In [ ]:
# 8. 結果確認：run一覧（best epoch / Val AUROC / Val AUPRC / Val loss / early stopping / 時間 / 成果物）
#    24 run が揃うまで条件選択は行わない（スクリプト側でも拒否する）。
!python scripts/12_taskA_stage1_report.py --runs-root "{RUNS_ROOT}" \
    --out "{RUNS_ROOT}/stage1_report.csv"

# 学習曲線（Loss と Validation AUROC）
import glob
import pandas as pd
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for h in sorted(glob.glob(f'{RUNS_ROOT}/*/seed*/history.csv')):
    df = pd.read_csv(h)
    label = '/'.join(h.split('/')[-3:-1])
    ax[0].plot(df.epoch, df.train_loss, alpha=.6)
    ax[0].plot(df.epoch, df.val_loss, '--', alpha=.6)
    ax[1].plot(df.epoch, df.val_auroc, alpha=.7, label=label)
ax[0].set_title('loss (solid=train, dashed=val)'); ax[0].set_xlabel('epoch')
ax[1].set_title('Validation AUROC'); ax[1].set_xlabel('epoch')
ax[1].legend(fontsize=6, ncol=2)
plt.tight_layout(); plt.show()